# Session 5 — Smart contracts are shared house rules

This is the **practice notebook**. For each idea: run the **Demo** (it already works), read **Keep this straight**, then do **YOUR TURN** with a friend — tip first, talk, fill `???` together. After your table finishes, the instructor talks about what the room found.

Every tip ties to a **real business** picture (invoice fraud, fund custody, club tab), not a slogan. There is no answer key here.

Lost? Open **Start here**: https://salihtutun.github.io/acct5105/fundamentals.html then re-run the Demo.

A smart contract is **not** a lawyer and **not** magic.

It is a vending machine: money in, rules run, snack (or coins) out.

By the end: mint, transfer, pause — the same buttons you press in Remix next time.



## Words we use today

Lost? Open **Start here**: https://salihtutun.github.io/acct5105/fundamentals.html

Read this once. If a later cell uses the word, come back here.

| Word | In this class it means |
|---|---|
| **Smart contract** | A vending machine: money in, posted rules run, snack (or coins) out. |
| **ERC-20** | The same buttons on every soda machine so one wallet app works. |
| **Mint** | The banker prints. Only the named minter may do this. |
| **Cap** | The box lid. Supply cannot go above this number. |
| **Pause** | Freeze button. Transfers stop. Compliance / something’s wrong. |
| **Gas** | You paid for the *attempt*, even if the snack did not drop. |
| **Revert** | No snack, machine still hummed. State rolls back. Gas is still gone. |



## 1. Posted rules plus a shared counter

If Ava’s napkin says “I owe 500” and Ben’s sticky note says the same, Cara
does not know which counts. A **shared rule** (the box lid, or a contract)
is how strangers agree on the same numbers.

**Demo.** Only the banker prints. Cap is the box lid. Transfer moves coins.



In [ ]:
# ===== Demo — flight-simulator coin (same jobs as Remix) =====
# Every line has a plain-English comment — students are not Python experts.

class ClassCoin:  # the vending-machine template (posted house rules + shared counter)
    def __init__(self, symbol, cap, owner):  # bolt the machine together when we create it
        self.symbol = symbol  # nickname on the jersey (not the serial number)
        self.cap = cap  # box lid — total supply may never go above this
        self.minter = owner  # who is allowed to print new coins (the banker)
        self.pauser = owner  # who may hit the freeze switch
        self.paused = False  # start unfrozen so transfers are allowed
        self.balances = {}  # empty mailbox book: name → how many coins
        self.supply = 0  # how many coins exist so far (starts at zero)

    def mint(self, caller, to, amount):  # banker prints `amount` into `to`'s mailbox
        if caller != self.minter:  # only the named banker may print
            raise PermissionError("only the banker prints")  # alarm: wrong person at the printer
        if self.supply + amount > self.cap:  # would this print break the lid?
            raise ValueError("box lid -- cap")  # alarm: lid does not stretch
        self.balances[to] = self.balances.get(to, 0) + amount  # add coins to their mailbox (0 if new)
        self.supply = self.supply + amount  # bump the shared counter of coins that exist
        print("mint", amount, "->", to, "supply", self.supply)  # show what just printed

    def transfer(self, caller, to, amount):  # move coins from caller to classmate
        if self.paused:  # freeze switch on?
            raise RuntimeError("frozen")  # alarm: no transfers while frozen
        if self.balances.get(caller, 0) < amount:  # do they have enough?
            raise ValueError("broke")  # alarm: empty pockets
        self.balances[caller] -= amount  # take coins out of the sender's mailbox
        self.balances[to] = self.balances.get(to, 0) + amount  # put those coins in the receiver's mailbox
        print(caller, "->", to, amount)  # show the move

    def pause(self, caller, value):  # flip the freeze switch True/False
        if caller != self.pauser:  # only the freeze person may flip it
            raise PermissionError("not the freeze person")  # alarm: wrong key for pause
        self.paused = value  # store the new freeze state


coin = ClassCoin("EAGE", 10000, "team")  # build one machine: ticker EAGE, lid 10000, team = banker
coin.mint("team", "Ava", 200)  # banker prints 200 into Ava's mailbox
coin.mint("team", "Ben", 200)  # banker prints 200 into Ben's mailbox
coin.transfer("Ava", "Ben", 6)  # Ava sends 6 to Ben (two mailbox updates)
print("Ava", coin.balances.get("Ava"), "Ben", coin.balances.get("Ben"))  # expect Ava 194, Ben 206



**What you just saw.** A contract is posted house rules plus a shared counter — not a lawyer, not magic.

### Keep this straight

**If you remember one thing.** Minter prints. Cap is the lid. Pause freezes transfers. Failed calls still cost gas on Ethereum.

**Common mix-up.** “Smart” means the contract is wise. It is only as wise as the if-then you typed.

**Why Eagecoin cares.** Next session you press these exact buttons in Remix: mint, transfer, pause. Premint = your A2 cap. Decide who holds the keys *before* you deploy.



## YOUR TURN — trigger the three alarms

On Ethereum, a failed call still costs **gas** (you paid for the hum).
The **revert** rolls state back. Classroom Python just raises. Name both.

Keep the Demo cell run so `coin` still exists. Fill each `???` with a call that **should fail**.



In [ ]:
# ===== YOUR TURN — three alarms (each line should FAIL) =====
# Fill every ???. Tip a friend. Hands up when all three alarms print.

print("--- cap ---")  # label: we are testing the box lid
try:  # attempt a bad mint; catch the alarm so the notebook keeps going
    ???  # mint far above the lid, e.g. coin.mint("team", "Ava", 1000000)
except Exception as exc:  # the vending machine rejected the print
    print("expected:", exc)  # show the lid alarm text

print("--- non-minter ---")  # label: we are testing the banker rule
try:  # attempt a print by the wrong person
    ???  # Ava tries to mint to herself, e.g. coin.mint("Ava", "Ava", 1)
except Exception as exc:  # banker rule rejected Ava
    print("expected:", exc)  # show the banker alarm text

print("--- pause ---")  # label: we are testing the freeze switch
coin.pause("team", True)  # freeze person turns transfers OFF
try:  # attempt a transfer while frozen
    ???  # anyone transfers, e.g. coin.transfer("Ben", "Cara", 1)
except Exception as exc:  # freeze rule rejected the move
    print("expected:", exc)  # show the frozen alarm text
print("paused?", coin.paused)  # confirm the switch is still True



## Exit

Who is your `minter`? `pauser`? What is the `cap`? If the minter loses
the key, then what? Write it for A2/A3. Next session we deploy.

**Never mainnet.**



## Check you understand (say it out loud)

1. Who is allowed to mint, and what stops them from printing 1,000,000?
2. Gas vs revert — which one can you still lose money on when nothing moved?
3. If the pauser key is lost, what happens to classmates who want to trade?

If an answer is a slogan (“decentralized,” “web3,” “the future of money”),
you do not understand it yet. Re-run the Demo cells. Still stuck? Open
**Start here** (https://salihtutun.github.io/acct5105/fundamentals.html),
then the concept handbook — cafeteria, key, and lemonade pictures, not a second lecture.

